# pyNTAVIS: training an SNN with snnTorch from three-axis AEDAT files

This notebook goes from a folder of labelled three-axis AEDAT recordings to a trained Spiking Neural Network (SNN):

1. Every recording is loaded with `Loaders.loadAEDAT()`.
2. It is cut into fixed-length windows with `Splitters.window_splitter()`. Each window takes the label of its recording.
3. Each window is converted into a `[time steps, addresses]` matrix of spike counts with `Functions.spikes_to_frames()`.
4. The prepared train and validation sets are saved to an output folder (`.npz` files).
5. A spiking network made of leaky integrate-and-fire (LIF) neurons is trained with **snnTorch**.

Times are given in microseconds (µs), the unit of the AEDAT timestamps after `Functions.adapt_timestamps()`.

## 0. Setup (Google Colab only)

If you opened this notebook in **Google Colab**, run this cell first. It downloads the pyNTAVIS repository, installs its modified version of `pyNAVIS` (instead of the PyPI release) and snnTorch, and moves into the `src/examples` folder. PyTorch is already installed in Colab.

To run it locally, install PyTorch and snnTorch in your environment first (`pip install torch snntorch`). In that case this cell only prints where `pyNAVIS` is loaded from.

In [ ]:
# Setup for Google Colab: use the pyNTAVIS version of pyNAVIS from GitHub, not the PyPI one
import os, sys

if 'google.colab' in sys.modules:
    if not os.path.isdir('/content/pyNTAVIS'):
        !git clone --depth 1 https://github.com/juamonsan/pyNTAVIS.git /content/pyNTAVIS
    !pip uninstall -y -q pyNAVIS
    !pip install -q /content/pyNTAVIS/src snntorch
    os.chdir('/content/pyNTAVIS/src/examples')

import pyNAVIS
print('pyNAVIS loaded from:', pyNAVIS.__file__)

## 1. Import the required libraries

In [ ]:
%matplotlib inline
import glob
import json
import os

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

import snntorch as snn
from snntorch import functional as SF
from snntorch import surrogate

from pyNAVIS import Functions, Loaders, MainSettings, Savers, SpikesFile, Splitters

## 2. Configuration

### Input folder structure

`SOURCE_PATH` must contain a `train` and a `validation` folder. Inside each of them there is **one subfolder per class**, named after the class, with the AEDAT recordings of that class:

```
SOURCE_PATH/
├── train/
│   ├── class_a/
│   │   ├── recording_01.aedat
│   │   └── recording_02.aedat
│   └── class_b/
│       └── ...
└── validation/
    ├── class_a/
    └── class_b/
```

Class names are taken from the `train` folder (sorted alphabetically). Keep **whole recordings** in either `train` or `validation`, never windows of the same recording in both: overlapping windows of one recording are almost identical, and mixing them would give an over-optimistic validation accuracy.

### Parameters

| Parameter | Meaning |
|---|---|
| `settings` | `MainSettings` of the recordings (the same values as in the three-axis tutorial) |
| `WINDOW_SIZE` | Length of each window, in µs |
| `TRAIN_STRIDE` | Time between consecutive training windows. Smaller than `WINDOW_SIZE` means overlapping windows (more training samples) |
| `VALIDATION_STRIDE` | Same for validation. Equal to `WINDOW_SIZE` so validation windows don't overlap |
| `TIME_STEP` | Width of each SNN time step, in µs. Each window has `WINDOW_SIZE / TIME_STEP` time steps |
| `MIN_SPIKES` | Windows with fewer spikes are discarded (silent parts of a recording) |
| `BINARY_FRAMES` | `True`: each input says only whether an address fired in a time step. `False`: number of spikes |

In [ ]:
SOURCE_PATH = 'snn_dataset/source'
OUTPUT_PATH = 'snn_dataset/prepared'

settings = MainSettings(num_channels=43, mono_stereo=MainSettings.THREE_AXIS, on_off_both=1,
                        address_size=4, ts_tick=1, bin_size=20000, verbose=False)

WINDOW_SIZE = 100_000        # 100 ms
TRAIN_STRIDE = 50_000        # 50 % overlap
VALIDATION_STRIDE = WINDOW_SIZE
TIME_STEP = 1_000            # 1 ms -> 100 time steps per window
MIN_SPIKES = 50
BINARY_FRAMES = False

NUM_ADDRESSES = settings.addresses_per_source * settings.num_sources
NUM_STEPS = int(np.ceil(WINDOW_SIZE / TIME_STEP))
print('Inputs per time step:', NUM_ADDRESSES, '| time steps per window:', NUM_STEPS)

## 3. (Optional) Create a demo dataset

**Skip this section if you already have your own dataset in `SOURCE_PATH`.** The cell does nothing when `SOURCE_PATH` exists.

To check that the whole pipeline works, it builds a tiny dataset from the example file `NTAS_Accel_3Axxis_43ch_ONOFF_addr4b_ts1.aedat`:

- class `xyz`: the original recording;
- class `yzx`: the same recording with the axes rotated (X data moved to the Y addresses, Y to Z and Z to X).

The first second of the recording goes to `train` (two files of 0.5 s per class) and the rest to `validation`. **These labels are artificial**: the demo only shows that the code runs, not how well an SNN classifies real signals.

In [ ]:
def save_segment(spikes_file, start, end, path, axis_shift=0):
    """Saves the spikes in [start, end) as an AEDAT file, optionally rotating the axes by axis_shift positions."""
    addresses = np.asarray(spikes_file.addresses)
    timestamps = np.asarray(spikes_file.timestamps)
    mask = (timestamps >= start) & (timestamps < end)
    block = settings.addresses_per_source
    shifted = (addresses[mask] + axis_shift * block) % (block * settings.num_sources)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    Savers.save_AEDAT(SpikesFile(shifted, timestamps[mask] - start), path, settings)


if not os.path.isdir(SOURCE_PATH):
    demo_file = Loaders.loadAEDAT('test_files/NTAS_Accel_3Axxis_43ch_ONOFF_addr4b_ts1.aedat', settings)
    Functions.adapt_timestamps(demo_file, settings)
    end = demo_file.max_ts + 1
    segments = {'train': [(0, 500_000), (500_000, 1_000_000)], 'validation': [(1_000_000, end)]}

    for class_name, axis_shift in [('xyz', 0), ('yzx', 1)]:
        for split, ranges in segments.items():
            for i, (start, stop) in enumerate(ranges):
                save_segment(demo_file, start, stop,
                             os.path.join(SOURCE_PATH, split, class_name, 'demo_' + str(i)), axis_shift)
    print('Demo dataset created in', SOURCE_PATH)
else:
    print('Using the existing dataset in', SOURCE_PATH)

## 4. Prepare the train and validation sets

For every recording, `prepare_split()`:

1. loads it with `Loaders.loadAEDAT()` and makes its timestamps start at 0 µs;
2. cuts it into windows with `Splitters.window_splitter()`. Timestamps inside each window are relative to the window start;
3. converts every window into a `[NUM_STEPS, NUM_ADDRESSES]` matrix with `Functions.spikes_to_frames()`.

Each split is saved in `OUTPUT_PATH` as a compressed `.npz` file with:

- `frames`: windows, shape `[num_windows, NUM_STEPS, NUM_ADDRESSES]`;
- `labels`: class index of each window;
- `recordings`: index of the recording each window comes from, and `recording_names`: the path of each recording. Both are used for the per-recording evaluation at the end.

The parameters used are saved in `config.json` next to them.

In [ ]:
CLASS_NAMES = sorted(d for d in os.listdir(os.path.join(SOURCE_PATH, 'train'))
                     if os.path.isdir(os.path.join(SOURCE_PATH, 'train', d)))
print('Classes:', CLASS_NAMES)


def prepare_split(split, stride):
    split_path = os.path.join(SOURCE_PATH, split)
    unknown = set(os.listdir(split_path)) - set(CLASS_NAMES)
    if unknown:
        print('Warning: classes not present in train are ignored:', sorted(unknown))

    frames, labels, recordings, recording_names = [], [], [], []
    for label, class_name in enumerate(CLASS_NAMES):
        files = sorted(glob.glob(os.path.join(split_path, class_name, '*.aedat')))
        num_windows = 0
        for path in files:
            spikes_file = Loaders.loadAEDAT(path, settings)
            Functions.adapt_timestamps(spikes_file, settings)
            windows = Splitters.window_splitter(spikes_file, settings, WINDOW_SIZE, stride=stride, min_spikes=MIN_SPIKES)

            for window in windows:
                frames.append(Functions.spikes_to_frames(window, settings, TIME_STEP, duration=WINDOW_SIZE,
                                                         start_ts=0, binary=BINARY_FRAMES))
                labels.append(label)
                recordings.append(len(recording_names))
            recording_names.append(os.path.relpath(path, SOURCE_PATH))
            num_windows += len(windows)
        print('  {:<10} {:<20} {:>4} recordings -> {:>6} windows'.format(split, class_name, len(files), num_windows))

    data = dict(frames=np.stack(frames), labels=np.array(labels), recordings=np.array(recordings),
                recording_names=np.array(recording_names), class_names=np.array(CLASS_NAMES))
    np.savez_compressed(os.path.join(OUTPUT_PATH, split + '.npz'), **data)
    return data


os.makedirs(OUTPUT_PATH, exist_ok=True)
train_data = prepare_split('train', TRAIN_STRIDE)
validation_data = prepare_split('validation', VALIDATION_STRIDE)

config = dict(num_channels=settings.num_channels, mono_stereo=settings.mono_stereo, on_off_both=settings.on_off_both,
              address_size=settings.address_size, ts_tick=settings.ts_tick, window_size_us=WINDOW_SIZE,
              train_stride_us=TRAIN_STRIDE, validation_stride_us=VALIDATION_STRIDE, time_step_us=TIME_STEP,
              min_spikes=MIN_SPIKES, binary_frames=BINARY_FRAMES, class_names=CLASS_NAMES)
with open(os.path.join(OUTPUT_PATH, 'config.json'), 'w') as f:
    json.dump(config, f, indent=2)

print('Train frames:', train_data['frames'].shape, '| validation frames:', validation_data['frames'].shape)

### Look at one prepared window

Each window is the input the SNN receives: one row per time step and one column per address. The vertical lines separate the X, Y and Z address blocks.

In [ ]:
example = train_data['frames'][0]
plt.figure(figsize=(10, 4))
plt.imshow(example.T, aspect='auto', origin='lower', cmap='hot', interpolation='nearest')
for axis in range(1, settings.num_sources):
    plt.axhline(axis * settings.addresses_per_source - 0.5, color='cyan', linewidth=0.8)
plt.xlabel('Time step (' + str(TIME_STEP) + ' µs)')
plt.ylabel('Address')
plt.title('Window 0 - class ' + CLASS_NAMES[train_data['labels'][0]])
plt.colorbar(label='No. of spikes')
plt.show()

## 5. Load the prepared data

The `.npz` files are loaded from `OUTPUT_PATH`, so once the data is prepared you can start the notebook from here. The windows are wrapped in PyTorch `DataLoader`s that return batches of shape `[batch, NUM_STEPS, NUM_ADDRESSES]`.

In [ ]:
BATCH_SIZE = 16

def load_split(split):
    data = np.load(os.path.join(OUTPUT_PATH, split + '.npz'))
    frames = torch.tensor(data['frames'], dtype=torch.float32)
    labels = torch.tensor(data['labels'], dtype=torch.long)
    return data, TensorDataset(frames, labels)

train_npz, train_set = load_split('train')
validation_npz, validation_set = load_split('validation')
CLASS_NAMES = list(train_npz['class_names'])

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
validation_loader = DataLoader(validation_set, batch_size=BATCH_SIZE, shuffle=False)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Train windows:', len(train_set), '| validation windows:', len(validation_set), '| device:', device)

## 6. Define the SNN

A simple feed-forward network of leaky integrate-and-fire neurons (`snn.Leaky`):

`NUM_ADDRESSES` inputs → fully connected → `NUM_HIDDEN` LIF neurons → fully connected → one LIF output neuron per class

The network is simulated for `NUM_STEPS` time steps. At every step it receives one row of the window. The predicted class is the output neuron that fires the most spikes. The membrane potential carries information between time steps, so the network can use the temporal structure of the signal. Spikes are not differentiable, so training uses a *surrogate gradient* (`fast_sigmoid`).

For harder tasks, try a recurrent layer (`snn.RLeaky`), more hidden neurons or a learnable decay rate (`learn_beta=True`).

In [ ]:
NUM_HIDDEN = 128
BETA = 0.9          # membrane decay rate per time step

class SpikingNet(nn.Module):
    def __init__(self, num_inputs, num_hidden, num_outputs, beta):
        super().__init__()
        spike_grad = surrogate.fast_sigmoid(slope=25)
        self.fc1 = nn.Linear(num_inputs, num_hidden)
        self.lif1 = snn.Leaky(beta=beta, spike_grad=spike_grad)
        self.fc2 = nn.Linear(num_hidden, num_outputs)
        self.lif2 = snn.Leaky(beta=beta, spike_grad=spike_grad)

    def forward(self, x):
        # x: [batch, time steps, inputs] -> output spikes: [time steps, batch, outputs]
        mem1 = self.lif1.init_leaky()
        mem2 = self.lif2.init_leaky()
        spk_out = []
        for step in range(x.shape[1]):
            spk1, mem1 = self.lif1(self.fc1(x[:, step]), mem1)
            spk2, mem2 = self.lif2(self.fc2(spk1), mem2)
            spk_out.append(spk2)
        return torch.stack(spk_out)


net = SpikingNet(NUM_ADDRESSES, NUM_HIDDEN, len(CLASS_NAMES), BETA).to(device)
print(net)

## 7. Train

The loss is `SF.ce_rate_loss()`: the cross-entropy of the spike count of each output neuron, which pushes the correct output neuron to fire more than the others. After every epoch the model is evaluated on the validation windows, and the model with the lowest validation loss so far is saved to `OUTPUT_PATH/snn_model.pt`. The loss is a better selection criterion than the accuracy, which often reaches its maximum early and then stops changing.

`NUM_EPOCHS`, `LEARNING_RATE` and `BATCH_SIZE` (section 5) are starting values. What matters is the total number of weight updates (`NUM_EPOCHS × train windows / BATCH_SIZE`): with very few windows, use a small batch or more epochs. With a large dataset, increase `BATCH_SIZE` to train faster.

In [ ]:
NUM_EPOCHS = 20
LEARNING_RATE = 2e-3

torch.manual_seed(0)
optimizer = torch.optim.Adam(net.parameters(), lr=LEARNING_RATE)
loss_fn = SF.ce_rate_loss()


def run_epoch(loader, train):
    net.train(train)
    total_loss, total_correct, total = 0.0, 0.0, 0
    with torch.set_grad_enabled(train):
        for frames, labels in loader:
            frames, labels = frames.to(device), labels.to(device)
            spk_out = net(frames)
            loss = loss_fn(spk_out, labels)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * len(labels)
            total_correct += SF.accuracy_rate(spk_out, labels) * len(labels)
            total += len(labels)
    return total_loss / total, total_correct / total


history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_loss = float('inf')
for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = run_epoch(train_loader, train=True)
    val_loss, val_acc = run_epoch(validation_loader, train=False)
    for key, value in zip(history, [train_loss, train_acc, val_loss, val_acc]):
        history[key].append(value)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch + 1
        torch.save(net.state_dict(), os.path.join(OUTPUT_PATH, 'snn_model.pt'))
    print('Epoch {:>3} | train loss {:.3f} acc {:.3f} | validation loss {:.3f} acc {:.3f}'.format(
        epoch + 1, train_loss, train_acc, val_loss, val_acc))

print('Best model: epoch {} (validation loss {:.3f}, accuracy {:.3f})'.format(
    best_epoch, best_val_loss, history['val_acc'][best_epoch - 1]))

## 8. Training curves

In [ ]:
epochs = np.arange(1, NUM_EPOCHS + 1)
fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(11, 4))
ax_loss.plot(epochs, history['train_loss'], label='Train')
ax_loss.plot(epochs, history['val_loss'], label='Validation')
ax_loss.set_xlabel('Epoch'); ax_loss.set_ylabel('Loss'); ax_loss.set_title('Loss'); ax_loss.legend()
ax_acc.plot(epochs, history['train_acc'], label='Train')
ax_acc.plot(epochs, history['val_acc'], label='Validation')
ax_acc.set_xlabel('Epoch'); ax_acc.set_ylabel('Accuracy'); ax_acc.set_title('Accuracy (per window)'); ax_acc.legend()
ax_acc.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

## 9. Per-recording evaluation

The best model (lowest validation loss) is loaded again and every validation window is classified. The windows of each recording are then combined by **majority vote** to get one prediction per recording, which is usually more reliable than the prediction of a single window.

In [ ]:
net.load_state_dict(torch.load(os.path.join(OUTPUT_PATH, 'snn_model.pt'), map_location=device))
net.eval()

with torch.no_grad():
    window_predictions = torch.cat([net(frames.to(device)).sum(dim=0).argmax(dim=1).cpu()
                                    for frames, _ in validation_loader]).numpy()

labels = validation_npz['labels']
recordings = validation_npz['recordings']
recording_names = validation_npz['recording_names']
print('Window accuracy: {:.3f}\n'.format(np.mean(window_predictions == labels)))

correct_recordings = 0
for i, name in enumerate(recording_names):
    mask = recordings == i
    if not mask.any():
        continue
    votes = np.bincount(window_predictions[mask], minlength=len(CLASS_NAMES))
    predicted, true = votes.argmax(), labels[mask][0]
    correct_recordings += predicted == true
    print('{:<45} true: {:<12} predicted: {:<12} ({} of {} windows)'.format(
        name, CLASS_NAMES[true], CLASS_NAMES[predicted], votes[predicted], mask.sum()))

num_recordings = len(np.unique(recordings))
print('\nRecording accuracy: {} of {} ({:.3f})'.format(correct_recordings, num_recordings, correct_recordings / num_recordings))